In [13]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [14]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/UGC-7596_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/UGC-7596_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     359   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        58   (2094, 964)   float32   


In [15]:
image_cut = image[77:177, 0:1890]
image_error_cut = image_error[77:177, 0:1890]

In [16]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00189
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [17]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [18]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/PSF/Real_seeing_UGC7596.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [19]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

## Halpha

In [20]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/Halpha_fit.csv", index_col=0)

In [21]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.07396915326147181
0.2014081967692415
0.09126220165514369
0.10973707156528395
0.3098551477388577
0.3175419795028448
0.40854357572260946
0.4091445731648095
0.16532986158651966
0.16811405508782162
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       1.167746e-03  1.039055e-22  9.114502e-25  8.244566e-26   
Green      5.750451e-17  1.563747e-03  3.571278e-11  4.606521e-14   
Purple     1.726176e-02  1.561306e+01  6.050333e+01  6.710052e+00   
Orange     1.374531e-13  2.808035e-01  1.481465e+01  7.472847e+01   
Chocolate  9.998157e+01  8.410457e+01  2.468202e+01  1.856148e+01   

                 Peak 5  
Blue       5.056567e-22  
Green      1.043539e-04  
Purple     7.105505e+00  
Orange     3.264523e-02  
Chocolate  9.286175e+01  


In [22]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.2218857245751202
0.5530432168944297
0.2756724648937593
0.3222120902501332
0.9039469373402542
0.9224204617009988
1.1889871847692013
1.1904823820385089
0.3322267090705454
0.3460472725859488
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       1.026549e-03  1.117952e-22  9.878414e-25  8.855933e-26   
Green      6.194282e-17  1.332341e-03  1.063855e-10  5.990893e-14   
Purple     1.754983e-02  1.613607e+01  5.828406e+01  7.245419e+00   
Orange     2.272132e-13  3.348123e-01  1.633575e+01  7.362248e+01   
Chocolate  9.998142e+01  8.352779e+01  2.538019e+01  1.913211e+01   

                 Peak 5  
Blue       6.093566e-22  
Green      2.601511e-04  
Purple     7.529705e+00  
Orange     4.764536e-02  
Chocolate  9.242239e+01  


In [23]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.3697130423618119
0.7424779046456146
0.468158241418253
0.5204526028255296
1.420269556753359
1.4387513232649891
1.8502530431232858
1.8519496774732223
0.5025863922546145
0.5369396538902877
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       7.088664e-04  1.353667e-22  1.204501e-24  1.058371e-25   
Green      7.439223e-17  9.520995e-04  1.633012e-08  1.150577e-13   
Purple     1.831841e-02  1.754014e+01  5.274239e+01  8.809712e+00   
Orange     5.743098e-13  4.877524e-01  2.033884e+01  7.070360e+01   
Chocolate  9.998097e+01  8.197116e+01  2.691877e+01  2.048669e+01   

                 Peak 5  
Blue       8.179379e-22  
Green      4.754806e-04  
Purple     8.280679e+00  
Orange     8.005702e-02  
Chocolate  9.163879e+01  


## HBeta

In [24]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/Hbeta_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.17445555899164616
0.19179004745616474
0.09507861762737209
0.09508670032852748
0.1352975109125909
0.13524595583011262
0.15995962128401167
0.1601043634893186
0.18354301421893982
0.1834960175286741
                 Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.555321e+01   0.003570   0.000386  9.654132e-05  7.694791e-03
Green      0.000000e+00  33.780217   0.004427  2.492078e-28  3.186907e+01
Purple     0.000000e+00   0.490598  52.356947  2.693036e+01  3.076418e-04
Orange     8.257005e-70   0.000008   1.757900  3.475243e+01  1.104026e-08
Chocolate  3.444679e+01  65.725606  45.880340  3.831711e+01  6.812292e+01


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.48842208702036416
0.5314607195280192
0.2847446040697668
0.284756562992234
0.4073515082998057
0.40718350561945166
0.47283169076998566
0.4731305464763689
0.35938473050238395
0.35922938064220433
                 Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.309243e+01   0.003653   0.000391  9.940323e-05  8.241070e-03
Green      0.000000e+00  32.942278   0.017723  4.630586e-18  3.040505e+01
Purple     0.000000e+00   1.217232  52.061532  2.679329e+01  6.222152e-03
Orange     5.721376e-66   0.000029   2.207493  3.432184e+01  1.083087e-07
Chocolate  3.690757e+01  65.836808  45.712861  3.888477e+01  6.958049e+01


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


0.7059520593626414
0.7516003075155238
0.4743951024477019
0.4744031270381585
0.6837904982929206
0.6834598737031526
0.7598009496130964
0.7599032913098884
0.525045827176719
0.5247581160896285
                 Peak 1     Peak 2     Peak 3        Peak 4     Peak 5
Blue       5.745324e+01   0.003868   0.000406  1.071662e-04   0.009148
Green      0.000000e+00  30.690928   0.099456  1.619989e-11  28.484102
Purple     0.000000e+00   3.449103  51.152139  2.640248e+01   0.070929
Orange     6.360286e-62   0.000165   3.369105  3.327622e+01   0.000001
Chocolate  4.254676e+01  65.855936  45.378894  4.032119e+01  71.435820


## NII

In [28]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/NII_fit.csv", index_col = 0)

In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.20087245604466733
0.19329628390511705
0.08859951305132516
0.08849310837287655
0.11175607079484912
0.11185620275714467
0.1253998255645879
0.12139826761694894
0.15799216212478862
0.15785034831448824
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.697489e+01   0.827035   0.326958   0.187930   1.227126
Green      1.689526e-01  37.725493   9.531461   3.054488  37.011015
Purple     1.006079e-09   8.710920  47.260057  13.082211   2.586738
Orange     2.702658e-13   0.002831   1.255863  46.791217   0.000335
Chocolate  2.285616e+01  52.733720  41.625661  36.884154  59.174786


In [30]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.565020254761239
0.543451286607683
0.2649212325288481
0.264625215435922
0.332269795049948
0.332506660262098
0.36228744832350607
0.35218068274444064
0.31594246721480446
0.315634056194697
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.544083e+01   0.832755   0.330754   0.195558   1.237599
Green      1.826503e-01  37.110542   9.698226   3.205122  36.596951
Purple     1.600208e-09   9.145998  46.424589  13.935856   2.984873
Orange     3.595769e-13   0.003502   1.546055  44.363769   0.000486
Chocolate  2.437651e+01  52.907203  42.000376  38.299694  59.180091


In [31]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.8014494813636953
0.7748005333144241
0.43815415370948196
0.4377382425456108
0.542715927414908
0.542883506910081
0.5616697331889946
0.5497343497098659
0.47256661735606975
0.4720891589737648
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       7.113703e+01   0.847332   0.339831   0.211423   1.258623
Green      2.222802e-01  35.558319  10.117511   3.542861  35.709614
Purple     4.049439e-09  10.276382  44.255878  15.935144   3.686386
Orange     6.814070e-13   0.005566   2.432426  39.139590   0.000828
Chocolate  2.864069e+01  53.312401  42.854354  41.170982  59.344548


## SII

In [32]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/SII_fit.csv", index_col = 0)

In [33]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.20022717385508707
0.21095082731860204
0.11000958342222893
0.11003315550920506
0.1417876557884849
0.14179559495619806
0.28593986621465206
0.14996429148028384
0.18318614520856605
0.18285499825466192
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.789824e+01  3.565618e-01  1.276336e-01  3.887804e-02   
Green      0.000000e+00  1.698520e+01  1.554919e-07  4.021498e-62   
Purple     3.038464e-13  1.749386e+01  5.128085e+01  1.809456e+01   
Orange     6.328149e-17  1.672482e-11  1.135304e-07  5.858720e+01   
Chocolate  3.210176e+01  6.516438e+01  4.859152e+01  2.327936e+01   

                 Peak 5  
Blue       5.847127e-01  
Green      1.381948e+01  
Purple     6.578988e+00  
Orange     3.168718e-12  
Chocolate  7.901682e+01  


In [34]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5666784921228334
0.5922648825194468
0.3277279954225234
0.3277022385397062
0.42354310455418787
0.4235348210764691
0.7844165117836766
0.4355477414660241
0.36870886176022954
0.36819892687036415
                 Peak 1        Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.597248e+01  3.606305e-01   0.128581  4.262480e-02  5.869373e-01
Green      0.000000e+00  1.603500e+01   0.000015  1.511440e-44  1.366645e+01
Purple     1.007054e-12  1.798908e+01  51.064410  1.969288e+01  7.295705e+00
Orange     6.918136e-17  2.029269e-11   0.008424  5.480729e+01  4.022232e-12
Chocolate  3.402752e+01  6.561529e+01  48.798570  2.545721e+01  7.845091e+01


In [35]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.8195339081759802
0.8457517933452776
0.5363942646318566
0.5360792038928914
0.698713071318442
0.6977664464429678
1.111948143534811
0.6868940149862859
0.5537037685671853
0.5532216101187242
                 Peak 1        Peak 2     Peak 3        Peak 4        Peak 5
Blue       6.078676e+01  3.715657e-01   0.130994  5.046073e-02  5.963024e-01
Green      0.000000e+00  1.354033e+01   0.000412  1.019655e-30  1.262929e+01
Purple     6.260115e-12  1.929119e+01  50.405730  2.284479e+01  8.518099e+00
Orange     8.626535e-17  3.197670e-11   0.166717  4.717547e+01  5.928581e-12
Chocolate  3.921324e+01  6.679692e+01  49.296147  2.992927e+01  7.825631e+01


## OIII

In [36]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC7596/UGC7596_1/Fit/OIII_fit.csv", index_col = 0)

In [37]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.21735869841033445
0.2220322171256393
0.1193640667551257
0.11943128337660111
0.16700875654855543
0.1676030468696077
0.14017221408037855
0.12581980049900646
0.21475121806694916
0.21472451002828596
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       68.631798   0.723775  2.530289e-01   0.192141   1.073063
Green       0.086419  25.286977  4.249512e+00   1.588405  25.089769
Purple      0.000833  13.865537  5.320237e+01  22.639979   6.832952
Orange      0.000000   0.000000  6.079559e-67  26.070699   0.000000
Chocolate  31.280949  60.123711  4.229509e+01  49.508775  67.004216


In [38]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6138769106090082
0.624987111870504
0.357252456626734
0.3574335350914203
0.4926223250209314
0.4940961596147786
0.429872314283533
0.3810697497508066
0.4290504191198109
0.4290994641630663
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       66.679694   0.728223  2.580387e-01   0.188375   1.083729
Green       0.093170  24.789917  4.375707e+00   1.574696  24.650266
Purple      0.000926  14.221047  5.235173e+01  22.552777   7.205420
Orange      0.000000   0.000000  1.336597e-12  27.254912   0.000000
Chocolate  33.226210  60.260813  4.301452e+01  48.429241  67.060585


In [39]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9061584155901994
0.9154572140697784
0.5919880299325142
0.5921258661001939
0.789498618337551
0.7910846855687265
0.7355078795111947
0.6438074478001138
0.6412850391879579
0.6413276833878094
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       62.378057   0.739948   0.270296   0.184588   1.103805
Green       0.109323  23.498951   4.702634   1.589453  23.760731
Purple      0.001168  15.162382  50.297981  23.017844   7.858614
Orange      0.000000   0.000000   0.002103  28.038773   0.000000
Chocolate  37.511452  60.598719  44.726987  47.169342  67.276850
